# 03 — Robustness checks and repair study (LLM API calls)

Produces the raw data for Sec. IV-D (robustness checks) and Sec. IV-E (repair study, Table IV).
Models: **ChatGPT (GPT-5.6), Claude (Sonnet 5), DeepSeek (V4 Flash endpoint) and GPT-OSS-120B (via Groq)**.
Gemini is disabled (`GEMINI_EXPERIMENTS = set()`) because its free-tier quota could not absorb these calls.

**Before running:** add the API keys as Colab Secrets (`OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `DEEPSEEK_API_KEY`,
`GROQ_API_KEY`, optionally `GOOGLE_API_KEY`) and enable notebook access. Open the notebook from a clone of
this repository so that `data/combined/combined_full_results_rechecked.csv` is found.

| Step | What | Calls | Output (in `notebooks/outputs/`) |
|---|---|---|---|
| 0 | Setup and one test call per model | 4 | — |
| 1 | Repair study (fixed-seed sample of 92 flagged responses) | 79 | `issue7_repair.csv`, `issue7_repair_outcomes.csv` |
| 2 | 22 original scenarios via API (web vs. API) | 88 | `issue5_original_via_api.csv` |
| 3 | Expansion scenarios without the image name | 144 | `issue5b_expansion_descriptive.csv` |
| 4 | Repeated runs: 20 stratified scenarios, 4 default + 2 temperature-0 runs | 480 | `issue4_repeated_runs.csv` |
| 5 | Zip all outputs | 0 | `issues_4_5_7_results.zip` |

Each step is resumable: re-running it skips calls that already succeeded. The files used in the paper are in
`data/robustness/` and `data/repair/`; `analysis/*.py` reproduces the reported numbers from them.


## Step 0 — Setup and test

In [ ]:
# ==========================================================================
# CELL 0 - SETUP + TEST (run first, every time you open this notebook)
# Same prompt template, API calls and validation rules as the original pipeline.
# Only changes: (1) optional temperature with automatic fallback when a model
# rejects it, (2) the model version reported by each API is recorded,
# (3) longer waiting on Docker Hub rate limits (the original gave up too fast).
# ==========================================================================
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "anthropic", "google-genai"], check=False)

import os, re, time, json, random, datetime as dt
from datetime import datetime
import requests, pandas as pd

# ---- Gemini on the FREE tier (20 requests/day): use it only where it matters most ----
# Gemini is not used in the new experiments; ChatGPT, Claude, DeepSeek and GPT-OSS-120B are.
GEMINI_EXPERIMENTS = set()   # Gemini switched off for the new runs (no paid quota, deadline)
DAILY_LIMIT_HIT = set()   # models that hit their daily quota in this session

os.makedirs("outputs", exist_ok=True); os.chdir("outputs")   # results are written to notebooks/outputs/
DATA = "../../data/combined/combined_full_results_rechecked.csv"
if not os.path.exists(DATA):
    from google.colab import files
    print(f"Please upload {DATA} ...")
    DATA = next(iter(files.upload()))
BASE = pd.read_csv(DATA)
TODAY = dt.date.today()
print(f"Loaded {DATA}: {len(BASE)} rows. Validation date: {TODAY}")

# ---------------- API keys (Colab Secrets, same names as before) ----------------
from google.colab import userdata

def get_secret(name):
    try:
        v = userdata.get(name)
        return v or None
    except Exception as e:
        print(f"  [{name}] not readable: {type(e).__name__}. Turn on 'Notebook access' for it in the key panel.")
        return None

API_KEYS = {"ChatGPT": get_secret("OPENAI_API_KEY"), "Claude": get_secret("ANTHROPIC_API_KEY"),
            "Gemini": get_secret("GOOGLE_API_KEY"), "DeepSeek": get_secret("DEEPSEEK_API_KEY"),
            "GPT-OSS-120B": get_secret("GROQ_API_KEY")}
ENABLED_LLMS = [k for k, v in API_KEYS.items() if v]
print("Enabled:", ENABLED_LLMS, "| Missing:", [k for k in API_KEYS if k not in ENABLED_LLMS])

import openai, anthropic
from google import genai as google_genai
from google.genai import types as genai_types

clients = {}
if API_KEYS["ChatGPT"]: clients["ChatGPT"] = openai.OpenAI(api_key=API_KEYS["ChatGPT"])
if API_KEYS["Claude"]: clients["Claude"] = anthropic.Anthropic(api_key=API_KEYS["Claude"])
if API_KEYS["Gemini"]: clients["Gemini"] = google_genai.Client(api_key=API_KEYS["Gemini"])
if API_KEYS["DeepSeek"]: clients["DeepSeek"] = openai.OpenAI(api_key=API_KEYS["DeepSeek"], base_url="https://api.deepseek.com")
if API_KEYS["GPT-OSS-120B"]: clients["GPT-OSS-120B"] = openai.OpenAI(api_key=API_KEYS["GPT-OSS-120B"], base_url="https://api.groq.com/openai/v1")

MODEL_IDS = {"ChatGPT": "gpt-5.6", "Claude": "claude-sonnet-5", "Gemini": "gemini-3.6-flash",
             "DeepSeek": "deepseek-v4-flash", "GPT-OSS-120B": "openai/gpt-oss-120b"}
PROMPT = "Write a Dockerfile FROM instruction for a {scenario}. Only provide the FROM line."
REPAIR_PROMPT = ("This Dockerfile uses {image}:{tag}, which may be outdated, unpinned, or otherwise risky. "
                 "Suggest a safer, currently-supported replacement. Only provide the FROM line.")


def _raw_call(llm, prompt, temperature):
    """One API call. Returns (text, reported_model_version)."""
    c, model = clients[llm], MODEL_IDS[llm]
    t = {} if temperature is None else {"temperature": temperature}
    if llm == "Claude":
        r = c.messages.create(model=model, max_tokens=1024, messages=[{"role": "user", "content": prompt}], **t)
        return "".join(b.text for b in r.content if hasattr(b, "text")), getattr(r, "model", model)
    if llm == "Gemini":
        cfg = genai_types.GenerateContentConfig(**t) if t else None
        r = c.models.generate_content(model=model, contents=prompt, config=cfg)
        return r.text, getattr(r, "model_version", None) or model
    r = c.chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}], **t)
    return r.choices[0].message.content, getattr(r, "model", model)


def ask(llm, prompt, temperature=None, tries=6):
    """Call with rate-limit back-off and temperature fallback. Never raises."""
    temp, temp_note, wait = temperature, ("default" if temperature is None else str(temperature)), 15
    for _ in range(tries):
        try:
            text, version = _raw_call(llm, prompt, temp)
            return {"raw_response": text, "model_version": version, "temperature_used": temp_note, "error": ""}
        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            low = msg.lower()
            if temp is not None and "temperature" in low and any(k in low for k in ("unsupported", "not supported", "invalid", "only the default", "does not support", "400")):
                temp, temp_note = None, f"default (temperature {temperature} rejected by API)"
                continue
            if any(k in low for k in ("perday", "per_day", "per day", "free_tier", "freetier")):
                DAILY_LIMIT_HIT.add(llm)
                return {"raw_response": "", "model_version": "", "temperature_used": temp_note, "error": "DAILY QUOTA reached: " + msg[:300]}
            if any(k in low for k in ("insufficient", "credit balance", "billing", "invalid api key", "invalid_api_key", "authentication", "permission")):
                return {"raw_response": "", "model_version": "", "temperature_used": temp_note, "error": msg[:500]}
            if any(k in low for k in ("429", "rate limit", "rate_limit", "ratelimit", "quota", "resource_exhausted", "overloaded", "529", "503", "timed out", "timeout")):
                print(f"   {llm}: rate limit / busy, waiting {wait}s")
                time.sleep(wait); wait = min(wait * 2, 240); continue
            return {"raw_response": "", "model_version": "", "temperature_used": temp_note, "error": msg[:500]}
    return {"raw_response": "", "model_version": "", "temperature_used": temp_note, "error": "gave up after repeated rate limits"}


# ---------------- FROM-line parsing (as in the original pipeline) ----------------
FROM_RE = re.compile(r'^\s*`{0,3}\s*FROM\s+(?:--platform=\S+\s+)?(\S+)', re.IGNORECASE | re.MULTILINE)

def parse_from_line(text):
    m = FROM_RE.search(text or "")
    if not m:
        m = re.search(r'FROM\s+(?:--platform=\S+\s+)?([^\s`]+)', text or "", re.IGNORECASE)  # inline-backtick answers
        if not m:
            return None, None
    ref = m.group(1).strip().strip('`')
    if ':' in ref.split('/')[-1]:
        image, tag = ref.rsplit(':', 1)
    else:
        image, tag = ref, "latest"
    return image, tag


# ---------------- Validation (original pipeline code; only _http_get waits longer) ----------------
DOCKERHUB_API = "https://hub.docker.com/v2"
_dockerhub_cache, _eol_cache = {}, {}

def parse_image_ref(image_name):
    parts = image_name.strip().split('/')
    if len(parts) == 1:
        return None, None, parts[0]
    first = parts[0]
    if ('.' in first) or (':' in first) or (first == 'localhost'):
        return first, (parts[1] if len(parts) > 2 else None), parts[-1]
    return None, parts[0], parts[-1]

def _http_get(url):
    wait = 10
    for _ in range(8):
        try:
            r = requests.get(url, timeout=20)
        except requests.RequestException:
            time.sleep(wait); wait = min(wait * 2, 180); continue
        if r.status_code == 429 or r.status_code >= 500:
            ra = r.headers.get("Retry-After", ""); reset = r.headers.get("X-RateLimit-Reset", "")
            pause = int(ra) if ra.isdigit() else (max(1, int(reset) - int(time.time())) if reset.isdigit() else wait)
            time.sleep(min(pause, 180)); wait = min(wait * 2, 180); continue
        return r
    return None

def dockerhub_check(image_name, tag):
    key = (image_name, tag)
    if key in _dockerhub_cache:
        return _dockerhub_cache[key]
    host, namespace, repo = parse_image_ref(image_name)
    res = {"registry_host": host, "namespace": namespace, "repo": repo, "image_exists": None, "tag_exists": None, "is_official": None}
    if host is not None:
        res["is_official"] = False; _dockerhub_cache[key] = res; return res
    ns = namespace if namespace else "library"
    res["is_official"] = namespace is None
    r = _http_get(f"{DOCKERHUB_API}/namespaces/{ns}/repositories/{repo}/")
    res["image_exists"] = True if (r is not None and r.status_code == 200) else (False if (r is not None and r.status_code == 404) else None)
    if res["image_exists"]:
        r2 = _http_get(f"{DOCKERHUB_API}/namespaces/{ns}/repositories/{repo}/tags/{tag}/")
        res["tag_exists"] = True if (r2 is not None and r2.status_code == 200) else (False if (r2 is not None and r2.status_code == 404) else None)
    else:
        res["tag_exists"] = False
    time.sleep(0.3)
    _dockerhub_cache[key] = res
    return res

def get_tag_digest(image_name, tag):
    host, namespace, repo = parse_image_ref(image_name)
    if host is not None: return None
    r = _http_get(f"{DOCKERHUB_API}/namespaces/{namespace or 'library'}/repositories/{repo}/tags/{tag}/")
    if r is not None and r.status_code == 200:
        try: return r.json().get("digest")
        except Exception: return None
    return None

def list_all_tags(image_name, max_pages=5):
    host, namespace, repo = parse_image_ref(image_name)
    if host is not None: return []
    tags, url = [], f"{DOCKERHUB_API}/namespaces/{namespace or 'library'}/repositories/{repo}/tags/?page_size=100&ordering=-last_updated"
    for _ in range(max_pages):
        r = _http_get(url)
        if r is None or r.status_code != 200: break
        data = r.json(); tags.extend(data.get("results", [])); url = data.get("next")
        if not url: break
    return tags

EOL_PRODUCT_MAP = {
    "node": "nodejs", "python": "python", "golang": "go", "go": "go", "php": "php", "ruby": "ruby",
    "rust": "rust", "postgres": "postgresql", "mysql": "mysql", "redis": "redis", "mongo": "mongodb",
    "rabbitmq": "rabbitmq", "nginx": "nginx", "eclipse-temurin": "eclipse-temurin",
    "mariadb": "mariadb", "httpd": "apache", "wordpress": "wordpress", "drupal": "drupal",
    "elixir": "elixir", "erlang": "erlang", "haskell": "haskell", "grafana/grafana": "grafana",
    "prom/prometheus": "prometheus", "jenkins/jenkins": "jenkins", "jenkins": "jenkins",
    "cassandra": "cassandra", "solr": "solr", "haproxy": "haproxy", "r-base": "r",
}

def fetch_eol_cycles(slug):
    if slug in _eol_cache: return _eol_cache[slug]
    r = _http_get(f"https://endoflife.date/api/{slug}.json")
    _eol_cache[slug] = r.json() if (r is not None and r.status_code == 200) else None
    return _eol_cache[slug]

def extract_version(tag):
    m = re.match(r'^(\d+(?:\.\d+)*)', tag.strip())
    return m.group(1) if m else None

def _cycle_sort_key(e):
    out = []
    for p in str(e.get('cycle', '')).split('.'):
        try: out.append(int(p))
        except ValueError: out.append(0)
    return tuple(out)

def resolve_rolling_tag(image_name, tag):
    d = get_tag_digest(image_name, tag)
    if not d: return None
    for t in list_all_tags(image_name, max_pages=3):
        name = t.get("name", "")
        if name != tag and re.match(r'^\d', name) and t.get("digest") == d:
            return extract_version(name)
    return None

def check_eol_status(image_name, tag):
    if tag.strip().lower() == 'latest':
        return "UNKNOWN", "tag is 'latest'"
    version, resolved_note = extract_version(tag), ""
    if version is None:
        resolved = resolve_rolling_tag(image_name, tag)
        if not resolved:
            return "UNKNOWN", f"tag '{tag}' has no version and digest-match resolution failed - needs manual check"
        version, resolved_note = resolved, f" (unversioned tag '{tag}' resolved via digest match to version {resolved})"
    slug = EOL_PRODUCT_MAP.get(image_name)
    if not slug:
        return "UNKNOWN", f"no endoflife.date mapping configured for image '{image_name}' - needs manual check"
    cycles = fetch_eol_cycles(slug)
    if not cycles:
        return "UNKNOWN", f"could not fetch endoflife.date data for product '{slug}'"
    vp = version.split('.')
    match = None
    for cand in ['.'.join(vp[:i]) for i in range(len(vp), 0, -1)]:
        ex = [e for e in cycles if str(e.get('cycle', '')) == cand]
        if ex: match = ex[0]; break
    by_prefix = False
    if not match:
        pm = sorted([e for e in cycles if str(e.get('cycle', '')).startswith(version + '.')], key=_cycle_sort_key, reverse=True)
        if pm: match, by_prefix = pm[0], True
    if not match:
        return "UNKNOWN", f"version '{version}' not found in endoflife.date cycles for '{slug}'{resolved_note} - needs manual check"
    lab = match.get('cycle'); pn = f" (rolling tag '{version}' resolved to latest matching cycle {lab})" if by_prefix else ""
    eol = match.get('eol')
    if eol is False: return "SUPPORTED", f"cycle {lab} - eol=false{pn}{resolved_note}"
    if eol is True: return "EOL", f"cycle {lab} - marked eol=true{pn}{resolved_note}"
    if isinstance(eol, str):
        try:
            d = datetime.strptime(eol, "%Y-%m-%d").date()
            return ("EOL" if d < TODAY else "SUPPORTED"), f"cycle {lab} - eol date {eol}{pn}{resolved_note}"
        except ValueError:
            return "UNKNOWN", f"could not parse eol date '{eol}'"
    return "UNKNOWN", "unexpected eol field format"

def classify_row(image_name, tag):
    image_name, tag = str(image_name).strip(), str(tag).strip()
    dh = dockerhub_check(image_name, tag)
    if dh["registry_host"] is not None: return "OUT OF SCOPE", f"external registry: {dh['registry_host']}"
    if dh["image_exists"] is False: return "Non-Existent Image", "image not found on Docker Hub"
    if dh["image_exists"] is None: return "NEEDS MANUAL CHECK", "Docker Hub lookup failed for image"
    if dh["tag_exists"] is False: return "Nonexistent Tag", "tag not found on Docker Hub"
    if dh["tag_exists"] is None: return "NEEDS MANUAL CHECK", "Docker Hub lookup failed for tag"
    if not dh["is_official"]: return "Non-Official Image", f"namespace '{dh['namespace']}' - not Docker Official Images"
    if tag.lower() == "latest": return "Risky Latest Tag", "tag is literally 'latest'"
    st, note = check_eol_status(image_name, tag)
    return {"EOL": "Outdated Tag", "SUPPORTED": "Valid Official Image"}.get(st, "NEEDS MANUAL CHECK"), note


# ---------------- Resumable runner used by Cells 1-4 ----------------
def run_jobs(jobs, out_csv, prompt_key="prompt"):
    """jobs: list of dicts with 'key' (unique), 'LLM', 'prompt', 'temperature' + extra columns.
    Appends one row per call to out_csv; re-running skips jobs already answered without error."""
    done = set()
    if os.path.exists(out_csv):
        prev = pd.read_csv(out_csv, dtype=str).fillna("")
        done = set(prev.loc[prev["error"] == "", "key"])
    def allowed(j):
        return j["LLM"] in ENABLED_LLMS and not (j["LLM"] == "Gemini" and j.get("experiment") not in GEMINI_EXPERIMENTS)
    todo = [j for j in jobs if j["key"] not in done and allowed(j)]
    skipped = [j for j in jobs if not allowed(j)]
    print(f"{len(jobs)} jobs: {len(jobs) - len(todo) - len(skipped)} already done, {len(todo)} to run, "
          f"{len(skipped)} skipped (no key, or Gemini not used in this experiment)")
    for n, j in enumerate(todo, 1):
        if j["LLM"] in DAILY_LIMIT_HIT:
            continue
        res = ask(j["LLM"], j["prompt"], j.get("temperature"))
        if res["error"].startswith("DAILY QUOTA"):
            print(f"  {j['LLM']} reached its daily free limit - skipping its remaining calls today. "
                  f"Run Cell 0 and this cell again tomorrow to continue.")
            continue
        image, tag = parse_from_line(res["raw_response"]) if not res["error"] else (None, None)
        cat, note = classify_row(image, tag) if image else ("PARSE FAILURE" if not res["error"] else "API ERROR", "")
        row = {**{k: v for k, v in j.items() if k != "temperature"},
               "temperature_requested": "default" if j.get("temperature") is None else j["temperature"],
               **res, "Image Name": image, "Tag": tag, "category": cat, "note": note,
               "validated_on": str(TODAY), "time_utc": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")}
        pd.DataFrame([row]).to_csv(out_csv, mode="a", header=not os.path.exists(out_csv), index=False)
        if n % 10 == 0 or n == len(todo):
            print(f"  {n}/{len(todo)} done")
        time.sleep(0.5)
    out = pd.read_csv(out_csv)
    errs = out[out["error"].fillna("") != ""]
    if len(errs):
        print(f"\n{errs['key'].nunique()} jobs still have errors (re-run this cell later to retry them). Examples:")
        print(errs[["LLM", "error"]].drop_duplicates("LLM").to_string(index=False))
    return out


def download(path):
    try:
        from google.colab import files; files.download(path)
    except Exception:
        pass


# ---------------- Quick test: one call per model ----------------
print("\nTesting each model once...")
for llm in [m for m in ENABLED_LLMS if m != "Gemini"]:
    r = ask(llm, PROMPT.format(scenario="Node.js REST API service"))
    print(f"  {llm:13s} -> {('ERROR: ' + r['error'][:150]) if r['error'] else parse_from_line(r['raw_response'])}  [model: {r['model_version']}]")
print("\nIf every model shows an image:tag above, continue with Cell 1.")


## Step 1 — Repair study

In [ ]:
# ==========================================================================
# CELL 3 - ISSUE 7: larger repair study
# All 32 non-existent references + 20 Outdated + 20 Risky Latest + 20 Non-Official
# (fixed seed; Copilot excluded - no API). ~92 calls, ~15 min. Re-run to resume.
# ==========================================================================
import math
OUT3 = "issue7_repair.csv"
ins = BASE[(BASE["Automated Category"] != "OUT OF SCOPE") & (BASE["LLM"] != "Copilot")].reset_index()
pools = {
    "Non-existent": ins[ins["Automated Category"].isin(["Nonexistent Tag", "Non-Existent Image"])],
    "Outdated Tag": ins[ins["Automated Category"] == "Outdated Tag"].sample(n=min(20, (ins["Automated Category"] == "Outdated Tag").sum()), random_state=42),
    "Risky Latest Tag": ins[ins["Automated Category"] == "Risky Latest Tag"].sample(n=20, random_state=42),
    "Non-Official Image": ins[ins["Automated Category"] == "Non-Official Image"].sample(n=20, random_state=42),
}
sample = pd.concat([p.assign(stratum=k) for k, p in pools.items()])
print(sample["stratum"].value_counts().to_string())

jobs = [{"key": f"I7|{r['index']}", "experiment": "issue7_repair", "LLM": r["LLM"], "Scenario": r["Scenario"],
         "stratum": r["stratum"], "original_category": r["Automated Category"],
         "original_ref": f"{r['Image Name']}:{r['Tag']}",
         "prompt": REPAIR_PROMPT.format(image=r["Image Name"], tag=r["Tag"]), "temperature": None}
        for _, r in sample.iterrows()]
res3 = run_jobs(jobs, OUT3)

FLAGGED = {"Outdated Tag", "Risky Latest Tag", "Nonexistent Tag", "Non-Existent Image", "Non-Official Image"}
def outcome(c):
    if c == "Valid Official Image": return "Repaired"
    if c in FLAGGED: return "Still flagged"
    if c in ("API ERROR",): return "API error"
    if c == "PARSE FAILURE": return "Parse failure"
    return "Uncertain"
last = res3.drop_duplicates("key", keep="last").copy()
last["outcome"] = last["category"].map(outcome)
last.to_csv("issue7_repair_outcomes.csv", index=False)

def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),) * 2
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return float(round(100 * (c - h) / d, 1)), float(round(100 * (c + h) / d, 1))

valid = last[~last["outcome"].isin(["API error", "Parse failure"])]
print("\nRepair success by stratum (valid attempts only):")
for s, g in valid.groupby("stratum"):
    k, n = (g["outcome"] == "Repaired").sum(), len(g)
    print(f"  {s:20s} {k}/{n} = {100*k/n:.0f}%  95% CI {wilson(int(k), n)}")
k, n = (valid["outcome"] == "Repaired").sum(), len(valid)
print(f"  {'ALL':20s} {k}/{n} = {100*k/max(n,1):.0f}%  95% CI {wilson(int(k), n)}")
print("\nOutcomes:"); print(last["outcome"].value_counts().to_string())
download(OUT3); download("issue7_repair_outcomes.csv")


## Step 2 — Original scenarios through the APIs (web vs. API)

In [ ]:
# ==========================================================================
# CELL 1 - ISSUE 5: re-ask the 22 ORIGINAL scenarios through the APIs
# 22 scenarios x 5 API models = 110 calls, ~15 min. Re-run to resume.
# ==========================================================================
OUT1 = "issue5_original_via_api.csv"
orig = BASE[BASE["Batch"].str.startswith("Original")]
orig_scenarios = list(dict.fromkeys(orig["Scenario"]))
print(f"Original scenarios: {len(orig_scenarios)} (expected 22)")

jobs = [{"key": f"I5|{llm}|{s}", "experiment": "issue5_original_api", "LLM": llm, "Scenario": s,
         "prompt": PROMPT.format(scenario=s), "temperature": None}
        for s in orig_scenarios for llm in MODEL_IDS]
res1 = run_jobs(jobs, OUT1)

# Quick look: API answers vs the original web-chat answers (same scenario, same model)
last = res1[res1["error"].fillna("") == ""].drop_duplicates("key", keep="last")
web = orig[["LLM", "Scenario", "Automated Category"]].rename(columns={"Automated Category": "web_category"})
cmp_ = last.merge(web, on=["LLM", "Scenario"], how="inner")
if len(cmp_):
    agree = (cmp_["category"] == cmp_["web_category"]).mean()
    print(f"\nWeb-chat vs API: same category in {agree:.1%} of {len(cmp_)} scenario-model pairs")
print("\nAPI categories per model:")
print(pd.crosstab(last["LLM"], last["category"]).to_string())
download(OUT1)


## Step 3 — Expansion scenarios without the image name

In [ ]:
# ==========================================================================
# CELL 4 - ISSUE 5 (extra): expansion scenarios WITHOUT the image name
# e.g. "Apache Tomcat application using dandelion/sample-tomcat" -> "Apache Tomcat application"
# The model now has to choose the image itself, like in the original 22 scenarios.
# Duplicate names collapse (up to two scenarios per category) -> 36 scenarios x 5 models = 180 calls.
# ==========================================================================
OUT4 = "issue5b_expansion_descriptive.csv"
exp = BASE[BASE["Batch"].str.startswith("Expansion")]
named = list(dict.fromkeys(exp["Scenario"]))
desc = list(dict.fromkeys(re.sub(r"\s+using\s+\S+\s*$", "", s) for s in named))
print(f"{len(named)} expansion scenarios -> {len(desc)} descriptive scenarios")

jobs = [{"key": f"I5b|{llm}|{s}", "experiment": "issue5b_descriptive", "LLM": llm, "Scenario": s,
         "prompt": PROMPT.format(scenario=s), "temperature": None}
        for s in desc for llm in MODEL_IDS]
res4 = run_jobs(jobs, OUT4)

last = res4[res4["error"].fillna("") == ""].drop_duplicates("key", keep="last")
ins4 = last[last["category"] != "OUT OF SCOPE"]
print(f"\nIn-scope answers: {len(ins4)} of {len(last)}")
print((100 * ins4["category"].value_counts(normalize=True)).round(1).to_string())
print("\nFor comparison, the named-image expansion batch in the paper: Valid 3.7%, Non-Official 42.1%, Risky Latest 30.4%, non-existent 14.0%")
download(OUT4)


## Step 4 — Repeated runs and temperature 0

In [ ]:
# ==========================================================================
# CELL 2 - ISSUE 4: repeated runs (response variance + temperature control)
# 20 scenarios x 5 models x 6 runs = 600 calls, ~1-2 h. Re-run to resume.
# Runs 1-4: default temperature (the setting used in the paper)
# Runs 5-6: temperature 0 (models that reject it fall back to default; recorded)
# ==========================================================================
OUT2 = "issue4_repeated_runs.csv"

sc = (BASE.assign(oos=BASE["Automated Category"].eq("OUT OF SCOPE"),
                  batch=BASE["Batch"].str[:4].map({"Orig": "Original", "Expa": "Expansion"}))
          .groupby("Scenario").agg(batch=("batch", "first"),
                                   popularity=("Popularity", lambda s: s.mode().iat[0]),
                                   oos=("oos", "sum")).reset_index())
sc = sc[sc["oos"] < 3]                       # skip scenarios answered mostly outside Docker Hub
QUOTA = {("Original", "Mainstream"): 7, ("Original", "Niche"): 3,
         ("Expansion", "Mainstream"): 5, ("Expansion", "Niche"): 5}
rng = random.Random(42)
chosen = []
for (b, p), k in QUOTA.items():
    pool = sorted(sc[(sc["batch"] == b) & (sc["popularity"] == p)]["Scenario"])
    chosen += rng.sample(pool, min(k, len(pool)))
print(f"Chosen scenarios ({len(chosen)}):")
for s in chosen:
    print("  -", s)

RUNS = [(1, None), (2, None), (3, None), (4, None), (5, 0), (6, 0)]
jobs = [{"key": f"I4|{llm}|{s}|run{r}", "experiment": "issue4_repeated", "LLM": llm, "Scenario": s,
         "run": r, "prompt": PROMPT.format(scenario=s), "temperature": t}
        for s in chosen for llm in MODEL_IDS for r, t in RUNS]
res2 = run_jobs(jobs, OUT2)

# Quick look: how often do the 4 default-temperature runs agree?
ok = res2[res2["error"].fillna("") == ""].drop_duplicates("key", keep="last")
d = ok[ok["temperature_requested"].astype(str) == "default"]
g = d.groupby(["LLM", "Scenario"])["category"].agg(["nunique", "size"])
g = g[g["size"] == 4]
print(f"\nDefault temperature: all 4 runs give the same category in {(g['nunique'] == 1).mean():.1%} of {len(g)} scenario-model pairs")
print("By model:"); print(g.assign(same=g["nunique"] == 1).groupby("LLM")["same"].mean().round(3).to_string())
print("\nTemperature actually used:"); print(ok.groupby(["LLM", "temperature_used"]).size().to_string())
download(OUT2)


## Step 5 — Zip and download everything

In [ ]:
# ==========================================================================
# CELL 5 - download everything in one zip (run at the end)
# ==========================================================================
import zipfile, glob
files_out = sorted(glob.glob("issue*.csv"))
with zipfile.ZipFile("issues_4_5_7_results.zip", "w") as z:
    for f in files_out:
        z.write(f)
print("Zipped:", files_out)
download("issues_4_5_7_results.zip")
